In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [3]:
IMG_SIZE = 128
BATCH_SIZE = 16
NUM_CLASSES = 7

In [4]:
#Load Dataset

train_df = pd.read_csv("Datasets/train.csv")
valid_df = pd.read_csv("Datasets/val.csv")
test_df = pd.read_csv("Datasets/test.csv")

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)
metadata = pd.read_csv("Datasets/HAM10000_metadata.csv")

metadata.head()

(7010, 8)
(1502, 8)
(1503, 8)


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [5]:
image_dir1 = "Datasets/HAM10000_images_part_1"
image_dir2 = "Datasets/HAM10000_images_part_2"

image_path = {}

for folder in [image_dir1, image_dir2]:

    for file in os.listdir(folder):

        image_id = file.split(".")[0]

        image_path[image_id] = os.path.join(folder, file)

metadata["path"] = metadata["image_id"].map(image_path)

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0027419.jpg
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0025030.jpg
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0026769.jpg
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0025661.jpg
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,Datasets/HAM10000_images_part_2\ISIC_0031633.jpg


In [6]:
print("Missing Paths :", metadata["path"].isna().sum())

metadata[["image_id", "path"]].head()

Missing Paths : 0


,image_id,path
0,ISIC_0027419,Datasets/HAM10000_images_part_1\ISIC_0027419.jpg
1,ISIC_0025030,Datasets/HAM10000_images_part_1\ISIC_0025030.jpg
2,ISIC_0026769,Datasets/HAM10000_images_part_1\ISIC_0026769.jpg
3,ISIC_0025661,Datasets/HAM10000_images_part_1\ISIC_0025661.jpg
4,ISIC_0031633,Datasets/HAM10000_images_part_2\ISIC_0031633.jpg


In [7]:
encoder = LabelEncoder()

metadata["label"] = encoder.fit_transform(metadata["dx"])

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0027419.jpg,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0025030.jpg,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0026769.jpg,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0025661.jpg,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,Datasets/HAM10000_images_part_2\ISIC_0031633.jpg,2


In [8]:
train_df, temp_df = train_test_split(

    metadata,

    test_size=0.30,

    stratify=metadata["label"],

    random_state=42
)

val_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["label"],

    random_state=42
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(7010, 9)
(1502, 9)
(1503, 9)


In [9]:
train_df["label"] = train_df["label"].astype(str)
val_df["label"] = val_df["label"].astype(str)
test_df["label"] = test_df["label"].astype(str)

In [10]:
train_df["path"] = train_df["path"].astype(str)
val_df["path"] = val_df["path"].astype(str)
test_df["path"] = test_df["path"].astype(str)

In [11]:
classes = np.unique(train_df["label"])

weights = compute_class_weight(

    class_weight="balanced",

    classes=classes,

    y=train_df["label"]
)

class_weights = dict(zip(range(len(classes)), weights))

print(class_weights)

{0: 4.37305053025577, 1: 2.7817460317460316, 2: 1.3022478172023035, 3: 12.36331569664903, 4: 1.285530900421786, 5: 0.21338772031292808, 6: 10.115440115440116}


In [12]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    width_shift_range=0.1,

    height_shift_range=0.1,

    zoom_range=0.2,

    horizontal_flip=True,

    vertical_flip=True
)

test_datagen = ImageDataGenerator(

    rescale=1./255
)

In [13]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [14]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    horizontal_flip=True,

    vertical_flip=True,

    zoom_range=0.2,

    brightness_range=[0.8,1.2]

)

test_datagen = ImageDataGenerator(

    rescale=1./255

)

In [15]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True

)

Found 7010 validated image filenames belonging to 7 classes.


# Deep CNN


In [16]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    Dense,
    Flatten,
    GlobalAveragePooling2D,
    Input
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.applications import (
    MobileNetV2,
    EfficientNetB0,
    ResNet50,
    DenseNet121
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *

deep_cnn = Sequential([

    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    Conv2D(32, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    GlobalAveragePooling2D(),

    Dense(512, activation="relu"),
    Dropout(0.5),

    Dense(256, activation="relu"),
    Dropout(0.3),

    Dense(NUM_CLASSES, activation="softmax")

])

deep_cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 128, 128, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128, 128, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 64, 64, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 64, 64, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 64, 64, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 32, 32, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 32, 32, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 32, 32, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 16, 16, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 16, 16, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 16, 16, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 8, 8, 256)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 655,047 (2.50 MB)

 Trainable params: 654,087 (2.50 MB)

 Non-trainable params: 960 (3.75 KB)

In [18]:
from tensorflow.keras.optimizers import Adam

deep_cnn.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [19]:
history = deep_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    class_weight=class_weights
)

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 353s 786ms/step - accuracy: 0.3477 - loss: 2.0582 - val_accuracy: 0.1711 - val_loss: 1.9517
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 1463s 3s/step - accuracy: 0.3686 - loss: 1.7761 - val_accuracy: 0.4647 - val_loss: 1.4558
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 368s 838ms/step - accuracy: 0.4027 - loss: 1.6666 - val_accuracy: 0.4414 - val_loss: 1.4854
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 375s 852ms/step - accuracy: 0.4524 - loss: 1.5254 - val_accuracy: 0.4487 - val_loss: 1.4497
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 3146s 7s/step - accuracy: 0.4530 - loss: 1.5475 - val_accuracy: 0.4587 - val_loss: 1.3906


In [21]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = deep_cnn.evaluate(train_generator, verbose=0)

val_loss, val_acc =deep_cnn.evaluate(val_generator, verbose=0)

test_loss, test_acc =deep_cnn.evaluate(test_generator, verbose=0)

pred = deep_cnn.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [22]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.46533524990081787, 0.45872169733047485, 0.46107783913612366, 0.7439789036699415, 0.46107784431137727, 0.5231062064362074]


In [25]:
deep_cnn.save("models/deep_cnn_early.keras")

# LEARNING RAATE 

In [26]:
deep_cnn = Sequential([

    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    Conv2D(32, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    GlobalAveragePooling2D(),

    Dense(512, activation="relu"),
    Dense(256, activation="relu"),

    Dense(NUM_CLASSES, activation="softmax")

])

In [27]:
from tensorflow.keras.optimizers import Adam

deep_cnn.compile(

    optimizer=Adam(learning_rate=0.0005),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [28]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_scheduler = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=2,

    min_lr=1e-7,

    verbose=1

)

In [29]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=4,

    restore_best_weights=True,

    verbose=1

)

In [32]:
history_lr = deep_cnn.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,
    class_weight=class_weights,

    callbacks=[lr_scheduler, early_stop]

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 186s 425ms/step - accuracy: 0.2543 - loss: 2.0094 - val_accuracy: 0.5692 - val_loss: 1.3420 - learning_rate: 5.0000e-04
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 166s 378ms/step - accuracy: 0.3350 - loss: 1.8784 - val_accuracy: 0.4541 - val_loss: 1.6055 - learning_rate: 5.0000e-04
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 324ms/step - accuracy: 0.3966 - loss: 1.7813
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
439/439 ━━━━━━━━━━━━━━━━━━━━ 158s 359ms/step - accuracy: 0.3966 - loss: 1.7813 - val_accuracy: 0.4834 - val_loss: 1.4636 - learning_rate: 5.0000e-04
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 156s 356ms/step - accuracy: 0.4048 - loss: 1.7008 - val_accuracy: 0.4767 - val_loss: 1.4236 - learning_rate: 2.5000e-04
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 158s 359ms/step - accuracy: 0.4251 - loss: 1.5688 - val_accuracy: 0.4660 - val_loss: 1.3354 - learning_rate: 2.5000e-04
Restoring model weights from the end of the best epo

In [33]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = deep_cnn.evaluate(train_generator, verbose=0)

val_loss, val_acc =deep_cnn.evaluate(val_generator, verbose=0)

test_loss, test_acc =deep_cnn.evaluate(test_generator, verbose=0)

pred = deep_cnn.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [ ]:
comparison_basic_lr = pd.DataFrame({

    "Metric":[
        "Train Accuracy",
        "Validation Accuracy",
        "Test Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Phase 4":[
    0.675749,  
    0.687084,  
    0.679308,  
    0.601153,  
    0.679308,  
    0.622486   
],
    "After Batch":[
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ]

})

comparison_basic_lr["Improvement"] = (

    comparison_basic_lr["After Batch"] -

    comparison_basic_lr["Phase 4"]

)

comparison_basic_lr

In [34]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.4681883156299591, 0.4660452604293823, 0.4491018056869507, 0.6978795400586457, 0.4491017964071856, 0.501995215412909]


In [35]:
deep_cnn.save("models/deep_cnn_lr.keras")

# SGD

In [36]:
deep_cnn_sgd = Sequential([

    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    Conv2D(32, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    GlobalAveragePooling2D(),

    Dense(512, activation="relu"),
    Dense(256, activation="relu"),

    Dense(NUM_CLASSES, activation="softmax")

])

In [37]:
from tensorflow.keras.optimizers import SGD

deep_cnn_sgd.compile(

    optimizer=SGD(
        learning_rate=0.01,
        momentum=0.9
    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [38]:
history_sgd = deep_cnn_sgd.fit(

    train_generator,

    validation_data=val_generator,
    
    class_weight=class_weights,
    epochs=5

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 162s 364ms/step - accuracy: 0.0544 - loss: 1.9781 - val_accuracy: 0.0113 - val_loss: 1.9780
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 162s 370ms/step - accuracy: 0.0328 - loss: 1.9452 - val_accuracy: 0.4960 - val_loss: 1.7593
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 157s 357ms/step - accuracy: 0.2219 - loss: 1.9863 - val_accuracy: 0.1112 - val_loss: 1.9385
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 156s 356ms/step - accuracy: 0.0644 - loss: 1.9741 - val_accuracy: 0.1099 - val_loss: 1.9252
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 939s 2s/step - accuracy: 0.0321 - loss: 1.9606 - val_accuracy: 0.0113 - val_loss: 1.9501


In [39]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = deep_cnn_sgd.evaluate(train_generator, verbose=0)

val_loss, val_acc =deep_cnn_sgd.evaluate(val_generator, verbose=0)

test_loss, test_acc =deep_cnn_sgd.evaluate(test_generator, verbose=0)

pred = deep_cnn_sgd.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [40]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.011554921977221966, 0.01131824217736721, 0.01131071150302887, 0.00012793220390002873, 0.011310711909514305, 0.00025300276639703053]


In [41]:
deep_cnn.save("models/deep_cnn_sgd.keras")

# RMSprop

In [46]:
deep_cnn_rmsprop = Sequential([

    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    Conv2D(32, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    GlobalAveragePooling2D(),

    Dense(512, activation="relu"),
    Dense(256, activation="relu"),

    Dense(NUM_CLASSES, activation="softmax")

])

In [47]:
from tensorflow.keras.optimizers import RMSprop

deep_cnn_rmsprop.compile(

    optimizer=RMSprop(
        learning_rate=0.001
    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [48]:
history_rmsprop = deep_cnn_rmsprop.fit(

    train_generator,

    validation_data=val_generator,class_weight=class_weights,

    epochs=5

)


Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 224s 508ms/step - accuracy: 0.3469 - loss: 1.8348 - val_accuracy: 0.4707 - val_loss: 1.6345
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 227s 517ms/step - accuracy: 0.4172 - loss: 1.8638 - val_accuracy: 0.5160 - val_loss: 1.3604
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 227s 517ms/step - accuracy: 0.3651 - loss: 1.8718 - val_accuracy: 0.2177 - val_loss: 1.5720
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 11124s 25s/step - accuracy: 0.3757 - loss: 1.5686 - val_accuracy: 0.3715 - val_loss: 1.5994
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 228s 519ms/step - accuracy: 0.4174 - loss: 1.5637 - val_accuracy: 0.3762 - val_loss: 1.4496


In [49]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = deep_cnn_rmsprop.evaluate(train_generator, verbose=0)

val_loss, val_acc =deep_cnn_rmsprop.evaluate(val_generator, verbose=0)

test_loss, test_acc =deep_cnn_rmsprop.evaluate(test_generator, verbose=0)

pred = deep_cnn_rmsprop.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [50]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.38787445425987244, 0.37616512179374695, 0.371257483959198, 0.7076895523679861, 0.3712574850299401, 0.4310763054537801]


In [51]:
deep_cnn_rmsprop.save("models/deep_cnn_rmsprop.keras")

# Batchwise Comparison

In [52]:
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 5

In [53]:
train_generator_8 = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=True

)

val_generator_8 = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

test_generator_8 = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [54]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense
deep_cnn_batch = Sequential([

    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    Conv2D(32, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    GlobalAveragePooling2D(),

    Dense(512, activation="relu"),
    Dense(256, activation="relu"),

    Dense(NUM_CLASSES, activation="softmax")

])

In [55]:
from tensorflow.keras.optimizers import RMSprop

deep_cnn_batch.compile(

    optimizer=RMSprop(
        learning_rate=0.001
    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [56]:
history = deep_cnn_batch.fit(

    train_generator_8,

    validation_data=val_generator,
    class_weight=class_weights,

    epochs=5

)

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 182s 819ms/step - accuracy: 0.1194 - loss: 1.9173 - val_accuracy: 0.3981 - val_loss: 1.9240
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 210s 953ms/step - accuracy: 0.4796 - loss: 1.8808 - val_accuracy: 0.5626 - val_loss: 1.3408
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 219s 995ms/step - accuracy: 0.4438 - loss: 1.7690 - val_accuracy: 0.3935 - val_loss: 1.7227
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 265s 1s/step - accuracy: 0.3633 - loss: 1.7782 - val_accuracy: 0.3855 - val_loss: 1.7073
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 216s 979ms/step - accuracy: 0.4313 - loss: 1.6253 - val_accuracy: 0.2503 - val_loss: 2.0649


In [57]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = deep_cnn_batch.evaluate(train_generator, verbose=0)

val_loss, val_acc =deep_cnn_batch.evaluate(val_generator, verbose=0)

test_loss, test_acc =deep_cnn_batch.evaluate(test_generator, verbose=0)

pred = deep_cnn_batch.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [58]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.27375179529190063, 0.25033289194107056, 0.24018628895282745, 0.6208202713506448, 0.24018629407850964, 0.2849719209015463]


In [59]:
deep_cnn_batch.save("models/deep_cnn_batch.keras")

# Hyperparameter Optimization

In [61]:
Adam(learning_rate=0.001)
Dense(512)
Dense(256)
Batch_Size = 16

In [62]:
!pip install keras-tuner

In [63]:
import keras_tuner as kt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    GlobalAveragePooling2D,
    Dense
)
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

In [64]:
def build_model(hp):

    model = Sequential([

        Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

        Conv2D(
            filters=hp.Choice(
                "filters1",
                [32, 64]
            ),
            kernel_size=(3,3),
            activation="relu"
        ),

        MaxPooling2D(2,2),

        Conv2D(
            filters=hp.Choice(
                "filters2",
                [64, 128]
            ),
            kernel_size=(3,3),
            activation="relu"
        ),

        MaxPooling2D(2,2),

        Conv2D(128,(3,3),activation="relu"),
        MaxPooling2D(2,2),

        GlobalAveragePooling2D(),

        Dense(
            units=hp.Choice(
                "dense_units",
                [128,256,512]
            ),
            activation="relu"
        ),

        Dense(NUM_CLASSES,activation="softmax")

    ])

    optimizer_name = hp.Choice(
        "optimizer",
        ["adam","sgd","rmsprop"]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        [0.01,0.001,0.0001]
    )

    if optimizer_name == "adam":
        optimizer = Adam(learning_rate)

    elif optimizer_name == "sgd":
        optimizer = SGD(learning_rate)

    else:
        optimizer = RMSprop(learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [65]:
tuner = kt.RandomSearch(

    build_model,

    objective="val_accuracy",

    max_trials=5,

    directory="deepcnn_tuning",

    project_name="deepcnn_hp"
)

Reloading Tuner from deepcnn_tuning\deepcnn_hp\tuner0.json


In [66]:
tuner.search(

    train_generator,

    validation_data=val_generator,

    epochs=5
)

In [67]:
best_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]

print(best_hps.values)

{'filters1': 64, 'filters2': 64, 'dense_units': 128, 'optimizer': 'rmsprop', 'learning_rate': 0.001}


In [68]:
best_model = tuner.hypermodel.build(
    best_hps
)

history = best_model.fit(

    train_generator,

    validation_data=val_generator,
    class_weight=class_weights,

    epochs=5
)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 243s 551ms/step - accuracy: 0.2153 - loss: 1.9175 - val_accuracy: 0.4387 - val_loss: 1.4311
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 256s 583ms/step - accuracy: 0.3183 - loss: 1.8327 - val_accuracy: 0.0826 - val_loss: 2.0307
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 262s 582ms/step - accuracy: 0.3395 - loss: 1.8254 - val_accuracy: 0.3562 - val_loss: 1.8226
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 260s 592ms/step - accuracy: 0.4108 - loss: 1.7714 - val_accuracy: 0.5173 - val_loss: 1.2200
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 266s 606ms/step - accuracy: 0.4458 - loss: 1.5558 - val_accuracy: 0.3868 - val_loss: 1.7277


In [69]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = best_model.evaluate(train_generator, verbose=0)

val_loss, val_acc =best_model.evaluate(val_generator, verbose=0)

test_loss, test_acc =best_model.evaluate(test_generator, verbose=0)

pred = best_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [70]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.37232524156570435, 0.3868175745010376, 0.3792415261268616, 0.7251992072629133, 0.37924151696606784, 0.44314702112337057]


In [71]:
best_model.save("best_model_deep.keras")

In [75]:
import pandas as pd

deepcnn_phase5 = pd.DataFrame({

    "Technique": [
        "Deep_cnn",
        "Early Stopping",
        "Learning Rate Scheduling",
        "SGD",
        "RMSprop",
        "Batch Size = 32",
        "Hyperparameter Tuning"
    ],

    "Train Accuracy": [
        0.46533524990081787,
        0.4681883156299591,
        0.4319543361663818,
        0.3721826195716858,
        0.3376176905632019,
        0.4744650602340698,
        0.3883023977279663
    ],



    "Validation Accuracy": [
         0.45872169733047485,
        0.4660452604293823,
        0.4410119771957397,
        0.38787445425987244,
        0.4844207644462585,
        0.356820704841614,
        0.391744327545166
    ],

    "Test Accuracy": [
        0.46107783913612366,
        0.4491018056869507,
        0.4398536205291748,
        0.37616512179374695,
        0.4799733638763428,
        0.3726546883583069,
        0.4919494271278381
    ],

    "Precision": [
        0.7439789036699415,
        0.6978795400586457,
        0.7991019177298748,
        0.7076895523679861,
        0.7518572307058556,
        0.4721277463313228,
        0.7827538288786149
    ],

    "Recall": [
        0.46107784431137727,
        0.4491017964071856,
        0.439853626081171,
        0.3712574850299401,
        0.4799733865602129,
        0.4726546906187625,
        0.4919494344644045
    ],

    "F1 Score": [
        0.5231062064362074,
        0.501995215412909,
        0.4256663450601256,
        0.4310763054537801,
        0.4880082993161069,
        0.4480985293357464,
        0.046790645631767
    ]
})

deepcnn_phase5.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)

,Technique,Train Accuracy,Validation Accuracy,Test Accuracy,Precision,Recall,F1 Score
0,Hyperparameter Tuning,0.388302,0.391744,0.491949,0.782754,0.491949,0.046791
1,RMSprop,0.337618,0.484421,0.479973,0.751857,0.479973,0.488008
2,Deep_cnn,0.465335,0.458722,0.461078,0.743979,0.461078,0.523106
3,Early Stopping,0.468188,0.466045,0.449102,0.697880,0.449102,0.501995
4,Learning Rate Scheduling,0.431954,0.441012,0.439854,0.799102,0.439854,0.425666
5,SGD,0.372183,0.387874,0.376165,0.707690,0.371257,0.431076
6,Batch Size = 32,0.474465,0.356821,0.372655,0.472128,0.472655,0.448099
